In [ ]:
import torch
import numpy as np
import cv2
from transformers import pipeline, utils
from PIL import Image
import polars as pl

import os

device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
from transformers import AutoImageProcessor, AutoModelForDepthEstimation

depth_processor = AutoImageProcessor.from_pretrained("depth-anything/Depth-Anything-V2-Small-hf")
depth_model = AutoModelForDepthEstimation.from_pretrained("depth-anything/Depth-Anything-V2-Small-hf").to(device)

In [ ]:
def depth_estimate_mask(image_number, seuil, folder_name):
    #this function crops the "image_number"th image of a folder "folder_name", by excluding all elements deeper than "seuil" 
    image = Image.open(f"../{folder_name}/{os.listdir(f"../{folder_name}")[image_number]}").convert("RGB")
    inputs = depth_processor(images=image, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = depth_model(**inputs)

    post_processed = depth_processor.post_process_depth_estimation(outputs, target_sizes=[image.size[::-1]])
    depth = post_processed[0]["predicted_depth"].cpu().numpy()
    depth_normalized = (depth - depth.min()) / (depth.max() - depth.min())

    n_bins = 10
    bin_edges = np.linspace(0, 1, n_bins + 1)
    bin_labels = [f"{bin_edges[i]:.1f}–{bin_edges[i+1]:.1f}" for i in range(n_bins)]
    pixel_counts = [
        int(((depth_normalized >= bin_edges[i]) & (depth_normalized < bin_edges[i + 1])).sum())
        for i in range(n_bins)
    ]
    pixel_counts[-1] = int(((depth_normalized >= bin_edges[-2]) & (depth_normalized <= 1.0)).sum())
    total_pixels = depth_normalized.size

    df = pl.DataFrame({
        "depth_range": bin_labels,
        "pixel_count": pixel_counts,
        "proportion": [pc / total_pixels for pc in pixel_counts]
    })
    df
    good = depth_normalized>= seuil
    rows, cols = np.where(good)

    im1_orig = np.array(image)
    h_orig, w_orig = im1_orig.shape[:2]
    masque = np.zeros((h_orig, w_orig), dtype=np.uint8)
    for r, c in zip(rows, cols):
        y, x = r * 1, c * 1
        masque[y:y + 1, x:x + 1] = 255
    n_labels, labels = cv2.connectedComponents(masque)
    tailles = [(labels == label).sum() for label in range(1, n_labels)]
    plus_grand_label = np.argmax(tailles) + 1
    masque_filtre = np.zeros_like(masque)
    masque_filtre[labels == plus_grand_label] = 255


    masque_orig = cv2.resize(masque_filtre, (w_orig, h_orig), interpolation=cv2.INTER_NEAREST)
    masque_3d = np.stack([masque_orig] * 3, axis=-1)
    image_masquee = np.where(masque_3d > 0, im1_orig, 0)

    nom_sortie = f"masque_{(os.listdir(f"../{folder_name}")[image_number]).replace(".jpg", "")}_{seuil}.png"# edit if the image is not in .jpg format 
    Image.fromarray(image_masquee).save(nom_sortie)

In [ ]:
for i in range (0, len(os.listdir(folder_name))):
    depth_estimate_mask(i, the_targeted_depth_threshold,folder_name)